In [222]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [223]:
# vars and consts

dt = 0.001
tau = 0.02

beta = np.exp(-dt/tau)

print("beta :", beta)



beta : 0.951229424500714


In [224]:
def LIF(mem, curr, beta, threshold):
    
    # spike
    spk = spike_fn(mem - threshold)

    mem = beta * mem + curr - spk * threshold

    return spk, mem

In [225]:
# surrogate spike function
class SurrogateGrad_fast_sig(torch.autograd.Function):

    @staticmethod
    def forward(ctx, mem_diff):

        ctx.save_for_backward(mem_diff)

        return (mem_diff >= 0).float()
    
    @staticmethod
    def backward(ctx, grad_output):
        
        (mem_diff,) = ctx.saved_tensors

        k = 10.0
        surr = 1.0 / (1.0 + k * torch.abs(mem_diff))**2

        return grad_output * surr

spike_fn = SurrogateGrad_fast_sig.apply

In [226]:
def define_random_neuron_connectivity_mask(num_neurons, connectivity, self_rec=False):

    mask = torch.rand(num_neurons, num_neurons)
    mask = (mask <= connectivity).float()

    if not self_rec:
        mask.fill_diagonal_(0)

    return mask

In [227]:
num_neurons = 100
m = define_random_neuron_connectivity_mask(num_neurons, 0.1)

print(torch.sum(m)/(num_neurons**2))

tensor(0.0977)


In [228]:
class RSNN(nn.Module):


    def __init__(self, in_features, num_neurons, out_features, beta, out_beta, threshold, mask):
        super().__init__()


        # constants
        self.beta = beta
        self.out_beta = out_beta
        self.threshold = threshold
        self.num_neurons = num_neurons
        self.out_features = out_features


        # input, recurrent weights, output
        self.W_in = nn.Linear(in_features, num_neurons, bias=False)

        self.W_rec = nn.Parameter(torch.randn(num_neurons, num_neurons)* 0.1)
        self.register_buffer("mask", mask.float())

        self.w_out = nn.Linear(num_neurons, out_features, bias=True)



    def forward(self, x):

        out_history = []

        # up to change based on task
        batch_size, num_steps, _ = x.shape

        mem = torch.zeros(batch_size, self.num_neurons)
        spk = torch.zeros(batch_size, self.num_neurons)
        out_mem = torch.zeros(batch_size, self.out_features)

        # effective W (W after applying mask)
        eff_W = self.W_rec * self.mask


        for t in range(num_steps):
            in_curr = self.W_in(x[:, t, :])
            rec_curr = torch.matmul(spk, eff_W.T)
            total_curr = rec_curr + in_curr

            # spike stuff
            spk = spike_fn(mem - self.threshold)
            mem = self.beta * mem + total_curr - spk * self.threshold

            # Task-agnostic Leaky Readout update (smooth continuous trace at every ms)
            out_mem = self.out_beta * out_mem + self.w_out(spk)
            out_history.append(out_mem)


        return torch.stack(out_history, dim=1)



In [229]:
def generate_batch_CR(batch_size=64, num_steps=30):
    X = torch.zeros(batch_size, num_steps, 2)
    labels = torch.randint(0, 2, (batch_size,))  # 0 or 1
    
    for i in range(batch_size):
        # Flash spikes on channel 0 or channel 1 during the first 5ms
        X[i, :5, labels[i]] = (torch.rand(5) > 0.3).float()
        # Add sparse background noise across all 30ms
        X[i] += (torch.rand(num_steps, 2) > 0.95).float()
        
    return X.clamp(0, 1), labels

In [230]:
# 3. Evidence Accumulation ("T-Maze") Data Generator
def generate_tmaze_batch(batch_size=64, cue_steps=60, delay_steps=30, decision_steps=10):
    total_steps = cue_steps + delay_steps + decision_steps
    X = torch.zeros(batch_size, total_steps, 3)  # Ch 0: Left, Ch 1: Right, Ch 2: Go Cue
    labels = torch.zeros(batch_size, dtype=torch.long)
    
    for i in range(batch_size):
        # Pick random spike counts for Left vs Right (guaranteed unequal)
        left_count = torch.randint(2, 8, (1,)).item()
        right_count = torch.randint(2, 8, (1,)).item()
        while left_count == right_count:
            right_count = torch.randint(2, 8, (1,)).item()
            
        # Scatter those spikes randomly across the 60ms cue window
        left_times = torch.randperm(cue_steps)[:left_count]
        right_times = torch.randperm(cue_steps)[:right_count]
        X[i, left_times, 0] = 1.0
        X[i, right_times, 1] = 1.0
        
        # Fire "Go Cue" on Channel 2 during the final decision window
        X[i, cue_steps + delay_steps:, 2] = 1.0
        
        # Label 0 if Left > Right, Label 1 if Right > Left
        labels[i] = 0 if left_count > right_count else 1
        
    return X, labels

In [231]:
# 3. Evidence Accumulation ("T-Maze") Data Generator
def generate_hard_tmaze_batch(batch_size=64, cue_steps=60, delay_steps=300, decision_steps=10):
    total_steps = cue_steps + delay_steps + decision_steps
    X = torch.zeros(batch_size, total_steps, 3)
    labels = torch.zeros(batch_size, dtype=torch.long)
    
    for i in range(batch_size):
        # Tight margin: difference of just 1 or 2 spikes!
        base = torch.randint(4, 7, (1,)).item()
        diff = torch.randint(1, 3, (1,)).item()
        if torch.rand(1).item() > 0.5:
            left_count, right_count = base + diff, base
            labels[i] = 0
        else:
            left_count, right_count = base, base + diff
            labels[i] = 1
            
        left_times = torch.randperm(cue_steps)[:left_count]
        right_times = torch.randperm(cue_steps)[:right_count]
        X[i, left_times, 0] = 1.0
        X[i, right_times, 1] = 1.0
        
        # Add continuous distractor noise on BOTH channels during the delay!
        noise = (torch.rand(delay_steps, 2) < 0.08).float()
        X[i, cue_steps:cue_steps + delay_steps, :2] = noise
        
        # Go Cue at the end
        X[i, cue_steps + delay_steps:, 2] = 1.0
        
    return X, labels

In [232]:
# setup
torch.manual_seed(42)
N = 40

threshold = 1.0

dt = 0.001
tau = 0.02

beta = np.exp(-dt/tau)

print("beta :", beta)

mask = define_random_neuron_connectivity_mask(N, 0.2, False)


rsnn = RSNN(in_features=3, num_neurons=N, out_features=2, beta=beta, out_beta=beta, threshold=threshold, mask=mask)

optimizer = torch.optim.Adam(rsnn.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

beta : 0.951229424500714


In [233]:
# train and test

for epoch in range(300):
    X_batch, y_batch = generate_hard_tmaze_batch(batch_size=64, cue_steps=60, delay_steps=100, decision_steps=10)
    
    optimizer.zero_grad()
    out_trace = rsnn(X_batch)                     # (64, 30, 40)
    logits = out_trace[:, -1, :]

    loss = criterion(logits, y_batch)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 1 == 0:
        acc = (logits.argmax(dim=1) == y_batch).float().mean().item() * 100
        # Verify mask stayed strictly enforced (masked weights are still 0)
        leaked_weights = (rsnn.W_rec * (1 - mask)).abs().max().item()
        print(f"Epoch {epoch+1:3d} | Loss: {loss.item():.4f} | Accuracy: {acc:5.1f}% | Masked Leak: {leaked_weights:.1f}")




Epoch   1 | Loss: 0.7040 | Accuracy:  50.0% | Masked Leak: 0.3
Epoch   2 | Loss: 0.8502 | Accuracy:  46.9% | Masked Leak: 0.3
Epoch   3 | Loss: 0.6997 | Accuracy:  53.1% | Masked Leak: 0.3
Epoch   4 | Loss: 0.6953 | Accuracy:  53.1% | Masked Leak: 0.3
Epoch   5 | Loss: 0.7497 | Accuracy:  51.6% | Masked Leak: 0.3
Epoch   6 | Loss: 0.7684 | Accuracy:  43.8% | Masked Leak: 0.3
Epoch   7 | Loss: 0.6987 | Accuracy:  46.9% | Masked Leak: 0.3
Epoch   8 | Loss: 0.7091 | Accuracy:  43.8% | Masked Leak: 0.3
Epoch   9 | Loss: 0.6910 | Accuracy:  53.1% | Masked Leak: 0.3
Epoch  10 | Loss: 0.6838 | Accuracy:  56.2% | Masked Leak: 0.3
Epoch  11 | Loss: 0.6971 | Accuracy:  56.2% | Masked Leak: 0.3
Epoch  12 | Loss: 0.6907 | Accuracy:  56.2% | Masked Leak: 0.3
Epoch  13 | Loss: 0.7204 | Accuracy:  51.6% | Masked Leak: 0.3
Epoch  14 | Loss: 0.6898 | Accuracy:  53.1% | Masked Leak: 0.3
Epoch  15 | Loss: 0.7003 | Accuracy:  48.4% | Masked Leak: 0.3
Epoch  16 | Loss: 0.6686 | Accuracy:  67.2% | Masked Le